## Tratamento de Dados

Importando as biliotecas:

In [ ]:
import pandas as pd 
import geopandas as gpd
import glob
import os

Iniciando com um dicionário com mapeamento das regiões e seus bairros correspondentes, e uma expressão que converte o bairro para região para facilitar o acesso. Serão úteis para regionalizar a análise futuramente.

In [ ]:
MAPEAMENTO_REGIOES = {
    'Zona Sul': ['Ipanema','Leblon','Botafogo','Copacabana','Gávea','Lagoa','Leme','Flamengo','Cosme Velho','Humaitá','São Conrado','Jardim Botânico','Rocinha','Vidigal','Urca','Laranjeiras','Catete','Glória','Santa Teresa','Joá'],
    'Centro': ['Centro','Cidade Nova','Santo Cristo','Gamboa','Saúde','Rio Comprido','Imperial de São Cristóvão','Praça da Bandeira','Catumbi','Estácio','Lapa'],
    'Zona Oeste': ['Campo Grande','Santa Cruz','Bangu','Realengo','Padre Miguel','Sepetiba','Guaratiba','Paciência','Santíssimo','Inhoaíba','Cosmos','Pedra de Guaratiba','Ilha de Guaratiba','Jabour','Jardim América','Magalhães Bastos','Senador Camará','Senador Vasconcelos','Vila Kennedy','Deodoro','Campo dos Afonsos','Vila Militar','Anchieta','Recreio dos Bandeirantes','Barra da Tijuca','Barra de Guaratiba','Barra Olímpica','Vargem Grande','Vargem Pequena','Camorim','Curicica','Gardênia Azul','Itanhangá','Jacarepaguá','Pechincha','Praça Seca','Taquara','Tanque','Anil','Cidade de Deus','Jardim Sulacap','Vila Valqueire','Freguesia (Jacarepaguá)'],
    'Zona Norte / Subúrbio': ['Madureira','Cascadura','Irajá','Pavuna','Marechal Hermes','Méier','Engenho de Dentro','Engenho da Rainha','Engenho Novo','Bonsucesso','Ramos','Olaria','Penha','Cordovil','Guadalupe','Rocha Miranda','Honório Gurgel','Coelho Neto','Bento Ribeiro','Del Castilho','Inhaúma','Acari','Água Santa','Grajaú','Vila Isabel','Andaraí','Tijuca','Alto da Boa Vista','Cachambi','Campinho','Encantado','Lins de Vasconcelos','Pilares','Piedade','Quintino Bocaiúva','Riachuelo','Rocha','Sampaio','Todos os Santos','Tomás Coelho','Turiaçu','Osvaldo Cruz','Barros Filho','Colégio','Costa Barros','Engenheiro Leal','Higienópolis','Parque Colúmbia','Parada de Lucas','Ricardo de Albuquerque','Vaz Lobo','Vicente de Carvalho','Vila da Penha','Vista Alegre','Vigário Geral','Complexo do Alemão','Jacarezinho','Manguinhos','Benfica','Caju','Mangueira','Maracanã','São Francisco Xavier','Jacaré','Brás de Pina','Cavalcanti','Penha Circular','Vasco da Gama','Maria da Graça','Bancários','Cacuia','Cocotá','Galeão','Jardim Carioca','Jardim Guanabara','Moneró','Pitangueiras','Portuguesa','Praia da Bandeira','Ribeira','Tauá','Zumbi','Maré','Abolição', 'Cidade Universitária', 'Freguesia (Ilha)']
}

# transforma o {regiao: [bairros]} em {bairro: regiao}
BAIRRO_TO_REGIAO = {bairro: regiao for regiao, lista in MAPEAMENTO_REGIOES.items() for bairro in lista}

Abaixo estão definidas as funções que tratam cada .csv de forma individual. No dicionário mapa_tratamento encontra-se o caminho do arquivo a qual cada função se refere.

In [ ]:
def tratar_ids(df):
    return df[['codbairro', 'bairro', 'ids']]

def tratar_lim_bairros(df):
    return df[['codbairro', 'bairro', 'limite']]

def tratar_lotacao_onibus(df):
    df['servico'] = df['servico'].astype(str)                                   # servico (linha) está como número mas é pra ser interpretado como string
    df = df[(df['servico'] !='795') & (~df['servico'].str.contains('LECD'))]
    df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y')                  # coloca a coluna data com valor de data no formato certo
    df = df[df['class_servico'] != 'Rodoviário']                                # remove do lotação onibus rodoviários
    return df

def tratar_sumario(df):
    df['data'] = pd.to_datetime(df['data'], format='%Y-%m-%d')
    df = df[df['km_planejada'] > 0]                                             # linha não era pra operar no dia
    df = df[['servico', 'data', 'perc_km_planejada']]                           # só precisa dessas 3 colunas
    return df

def tratar_routes(df):
    df = df.drop_duplicates(subset=['route_id', 'versao_modelo'])
    df = df[df['route_type'] != 702]
    df = df[~df['route_short_name'].str.contains('LECD|ESP01', na=False)]
    df['route_short_name'] = df['route_short_name'].str.lstrip('0')
    return df

def tratar_stops(df):
    return df.drop_duplicates(subset=['stop_id', 'versao_modelo'])

def tratar_route_stops(df):
    df = df[df['service_id'].isin(['D', 'D_REG', 'S', 'S_REG', 'U', 'U_REG'])]
    df = df.drop_duplicates(['stop_id', 'route_id', 'versao_modelo']).drop(['service_id'], axis=1)
    return df

mapa_tratamento = {
    'ids_por_bairro_rj.csv': tratar_ids,
    'limite_de_bairros.csv': tratar_lim_bairros,
    'lotacao_onibus.csv': tratar_lotacao_onibus,
    'sumario_servico_dia_tipo.csv': tratar_sumario,
    'routes.csv': tratar_routes,
    'stops.csv': tratar_stops,
    'route_stops.csv': tratar_route_stops,
}

Abaixo, esse for loop lê os arquivos na pasta *dados/* e realiza o tratamento adequado a cada um, aplicando as funções declaradas acima. No fim, armazena os DataFrames tratados no dicionário *datasets*. 

In [ ]:
datasets = {}

# daqui em diante ele vai tratar tudo e colocar no dicionário datasets criado ali em cima
arquivos = glob.glob('../dados/**/*.csv', recursive=True)

for caminho in arquivos:
    nome_arquivo = os.path.basename(caminho)

    nome_df = os.path.splitext(nome_arquivo)[0]
    
    df = pd.read_csv(caminho)
    
    if nome_arquivo in mapa_tratamento:
        df = mapa_tratamento[nome_arquivo](df)
    
        datasets[nome_df] = df

Criando a matriz de pares linha-bairro, que é o resultado do merge entre routes, stops e route_stops, e armazenando no dicionário *datasets* com a chave *'matriz_linha_bairro'*:

In [ ]:
def obter_matriz_linha_bairo(df_routes, df_stops, df_route_stops):
    df_pipeline = pd.merge(
        df_route_stops, 
        df_routes[['route_id', 'versao_modelo', 'route_short_name']], 
        on=['route_id', 'versao_modelo'], 
        how='inner'
    )

    df_pipeline = pd.merge(
        df_pipeline, 
        df_stops[['stop_id', 'versao_modelo', 'stop_lat', 'stop_lon']], 
        on=['stop_id', 'versao_modelo'], 
        how='inner'
    )

    df_pipeline = df_pipeline.drop_duplicates().drop(['route_id', 'stop_id', 'versao_modelo'], axis=1).rename(columns={'route_short_name': 'linha', 'stop_lat': 'latitude', 'stop_lon': 'longitude'})

    gdf_onibus = gpd.GeoDataFrame(
        df_pipeline,
        geometry=gpd.points_from_xy(df_pipeline.longitude, df_pipeline.latitude),
        crs="EPSG:4326"
    )

    gdf_bairros = gpd.read_file('../dados/Limite_de_Bairros.geojson')

    gdf_bairros = gdf_bairros.to_crs("EPSG:4326")

    resultado = gpd.sjoin(gdf_onibus, gdf_bairros, how="inner", predicate="intersects")

    matriz = (
        resultado[['nome', 'linha']]
        .drop_duplicates()
        .rename(columns={'nome': 'bairro'})
        .sort_values(['bairro', 'linha'])
        .reset_index(drop=True)
    )

    return matriz

datasets['matriz_linha_bairro'] = obter_matriz_linha_bairo(datasets['routes'], datasets['stops'], datasets['route_stops'])

Adicionando uma coluna na matriz de pares linha-bairro que classifica os bairros por região, conforme o MAPEAMENTO_REGIOES da segunda célula e, em seguida, salvando em csv:

In [ ]:
def tratar_matriz_lb(df):
    df['linha'] = df['linha'].astype(str)
    df['regiao'] = df['bairro'].map(BAIRRO_TO_REGIAO).fillna('Não identificado')# cria a coluna regiao mapeando pelo dicionário, e preenche os nan com "Não identificado"
    return df

datasets['matriz_linha_bairro'] = tratar_matriz_lb(datasets['matriz_linha_bairro'])

Salvando os arquivos tratados no formato csv. Eles serão lidos no próximo notebook, o *02_eda.ipynb*.

In [ ]:
def salvar_datasets(dicionario_dfs, mapa_chaves, pasta_destino='../dados_tratados', extras=None):

    if not os.path.exists(pasta_destino):
        os.makedirs(pasta_destino)
        print(f"Pasta '{pasta_destino}' criada")

    chaves_validas = [os.path.splitext(arq)[0] for arq in mapa_chaves.keys()]

    if extras:
        chaves_validas.extend(extras)

    for nome, df in dicionario_dfs.items():
        if nome in chaves_validas:
            caminho_arquivo = os.path.join(pasta_destino, f"{nome}_tratado.csv")
        
            try:
                df.to_csv(caminho_arquivo, index=False, encoding='utf-8')
                print(f"{nome}_tratado.csv salvo.")
            except Exception as e:
                print(f"Erro ao salvar o arquivo {nome}: {e}")

salvar_datasets(datasets, mapa_tratamento, extras=['matriz_linha_bairro'])